# 07 - Finding and Fixing the Loopholes

A systematic audit of everything built so far, done deliberately
adversarially: assume the pipeline has bugs, go looking for them, and
fix what's found rather than only what was originally planned.
Four real issues were found this way:

1. **No gate on the input data.** If someone points this project at a
   folder that's missing a file, has the wrong columns, or is
   otherwise corrupted, `preprocessing.py` would previously do its
   best silently and produce a master table that *looks* fine but
   isn't trustworthy. Fixed with `data_validation.py` - it inspects
   the raw folder and returns ACCEPT / ACCEPT_WITH_WARNINGS / REJECT
   with specific reasons, before any cleaning happens.
2. **A genuine threshold-leakage bug.** Notebook 06 selected the
   F1-optimal decision threshold using the TEST set's own labels, then
   reported F1 on that same test set - tuning a knob on the exam and
   then grading yourself with it. Fixed by selecting the threshold on
   a validation slice carved out of TRAINING data only.
3. **No guardrails at prediction time.** Nothing stopped a malformed
   order (missing field, impossible negative value) from being scored
   as if it were normal. Fixed with `inference_validation.py`, which
   rejects malformed input outright.
4. **A subtler guardrail bug, caught by testing the guardrail itself
   before trusting it**: the first version of the out-of-distribution
   check flagged genuinely high-risk orders (high seller historical
   delay rate) as "low confidence" purely for being unusual - exactly
   backwards, since that's the signal the model exists to catch. Fixed
   by giving bounded rate features a structural [0,1] check instead of
   a percentile-based "this is unusual" check.

In [1]:
import sys, os, json
sys.path.append("../src")
import pandas as pd
import numpy as np
import joblib
from sklearn.base import clone

from features import FEATURE_COLUMNS, TARGET_COLUMN
from modeling import stratified_split, evaluate, find_best_threshold, select_operating_threshold, NUMERIC, CATEGORICAL, risk_bucket
from data_validation import validate_raw_dataset, format_report
from inference_validation import build_reference_stats, save_reference_stats, predict_with_guardrails

MODEL_DIR = "../models"
os.makedirs(MODEL_DIR, exist_ok=True)

## 1. Validate the raw dataset before trusting it

This is the check that should have gated notebook 02 from the start.
Running it now, retroactively, on the real data:

In [2]:
report = validate_raw_dataset("../data/raw")
print(format_report(report))
assert report["verdict"] != "REJECT"

Verdict: ACCEPT  (23 checks, 0 failed, 0 warnings)

All checks passed cleanly.


All clean - as it should be, this is the genuine Olist dataset. The
point of this check isn't to catch a problem in data we already know
is fine, it's to catch a problem the NEXT time this pipeline is
pointed at something else. To prove the check actually works rather
than always saying OK, here it is run against a deliberately corrupted
copy: a required column removed, 50 duplicated order IDs, an items
table pointed at nonexistent orders and products, and negative prices.

In [3]:
import shutil
BAD_DIR = "/tmp/ontime_corrupted_demo"
shutil.rmtree(BAD_DIR, ignore_errors=True)
os.makedirs(BAD_DIR)
for f in os.listdir("../data/raw"):
    shutil.copy(f"../data/raw/{f}", f"{BAD_DIR}/{f}")

orders = pd.read_csv(f"{BAD_DIR}/olist_orders_dataset.csv")
orders = orders.drop(columns=["order_estimated_delivery_date"])       # missing required column
orders = pd.concat([orders.head(50), orders.head(50)])                # 50 duplicate order_ids
orders.to_csv(f"{BAD_DIR}/olist_orders_dataset.csv", index=False)

items = pd.read_csv(f"{BAD_DIR}/olist_order_items_dataset.csv").head(200).copy()
items["price"] = -items["price"]                                      # negative prices
items["order_id"] = "nonexistent_" + items.index.astype(str)           # orphaned foreign keys
items.to_csv(f"{BAD_DIR}/olist_order_items_dataset.csv", index=False)

products = pd.read_csv(f"{BAD_DIR}/olist_products_dataset.csv").head(5)  # gutted down to 5 rows
products.to_csv(f"{BAD_DIR}/olist_products_dataset.csv", index=False)

bad_report = validate_raw_dataset(BAD_DIR)
print(format_report(bad_report))
assert bad_report["verdict"] == "REJECT"
shutil.rmtree(BAD_DIR, ignore_errors=True)

Verdict: REJECT  (22 checks, 7 failed, 1 warnings)

[ FAIL ] schema:olist_orders_dataset.csv: Missing required column(s): ['order_estimated_delivery_date']
[ FAIL ] rows:olist_orders_dataset.csv: Only 100 rows found, expected at least 1,000.
[ FAIL ] rows:olist_order_items_dataset.csv: Only 200 rows found, expected at least 1,000.
[ FAIL ] rows:olist_products_dataset.csv: Only 5 rows found, expected at least 100.
[ FAIL ] duplicates:orders.order_id: 50 duplicate order_id values (each order must appear once).
[ FAIL ] referential:items->orders: 100.0% of order_items rows reference an order_id absent from orders.
[ FAIL ] referential:items->products: 100.0% of order_items rows reference a product_id absent from products.
[ warn ] values:items_negative: 100.00% of item rows have a negative price or freight value.


Correctly rejected, with specific, actionable reasons rather than a
generic failure - this is what stops a corrupted or mismatched dataset
from silently producing a plausible-looking but meaningless model.

## 2. The threshold-leakage bug, found and fixed

Notebook 06's threshold tuning did this:
```python
proba_test = best_xgb.predict_proba(X_test)[:, 1]
best_threshold, _ = find_best_threshold(y_test.values, proba_test)   # <- selected USING y_test
res_tuned_th = evaluate(best_xgb, X_test, y_test, threshold=best_threshold)  # <- then graded on y_test again
```
The threshold was fit to that exact test set's labels, then the model
was graded on those same labels with that threshold - which biases
the reported F1 optimistically. The fix: select the threshold using a
validation slice carved out of **training** data, and never let the
test set's labels influence anything except the final, one-time score.

In [4]:
df = pd.read_csv("../data/processed/ontime_model_ready.csv", parse_dates=["order_purchase_timestamp"])
train_df, test_df = stratified_split(df)
X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]

final_model = joblib.load(f"{MODEL_DIR}/ontime_final_model.joblib")
unfit_arch = clone(final_model)  # same tuned hyperparameters, not yet fit on anything

honest_threshold, val_f1 = select_operating_threshold(unfit_arch, X_train, y_train, val_frac=0.15)

proba_test = final_model.predict_proba(X_test)[:, 1]
leaky_threshold, _ = find_best_threshold(y_test.values, proba_test)  # the buggy way, kept only for comparison

res_default = evaluate(final_model, X_test, y_test, threshold=0.5)
res_leaky = evaluate(final_model, X_test, y_test, threshold=leaky_threshold)
res_honest = evaluate(final_model, X_test, y_test, threshold=honest_threshold)

pd.DataFrame({
    "default (0.50)": {k: round(v, 3) for k, v in res_default.items() if k in ("precision", "recall", "f1", "roc_auc")},
    "leaky-tuned (bug)": {k: round(v, 3) for k, v in res_leaky.items() if k in ("precision", "recall", "f1", "roc_auc")},
    "honest-tuned (fix)": {k: round(v, 3) for k, v in res_honest.items() if k in ("precision", "recall", "f1", "roc_auc")},
}).T

,precision,recall,f1,roc_auc
default (0.50),0.251,0.612,0.356,0.804
leaky-tuned (bug),0.354,0.419,0.384,0.804
honest-tuned (fix),0.311,0.483,0.378,0.804


The bias here turned out to be small (F1 0.383 leaky vs. 0.380 honest)
- reassuring, but not the point. The point is that "small in this
instance" isn't something you can know in advance, and the honest
number is the one that generalises to genuinely new data. **The honest
threshold is what the project now reports and uses.**

## 3. Inference-time guardrails

Reference statistics from the training data, used to sanity-check any
new order before it's scored. Rate-type features (bounded 0-1
historical delay rates) get a structural check instead of a
percentile check - see point 4 in the intro for why.

In [5]:
RATE_COLS = ["seller_historical_delay_rate", "category_historical_delay_rate", "customer_state_historical_delay_rate"]
stats = build_reference_stats(train_df, NUMERIC, CATEGORICAL, rate_cols=RATE_COLS)
save_reference_stats(stats, f"{MODEL_DIR}/reference_stats.json")
print(f"Reference stats built from {len(train_df):,} training orders, saved to models/reference_stats.json")

Reference stats built from 77,176 training orders, saved to models/reference_stats.json


Four scenarios, run through `predict_with_guardrails`:

In [6]:
risky_real = test_df.loc[test_df["seller_historical_delay_rate"] > 0.4].iloc[0][FEATURE_COLUMNS].to_dict()

scenarios = {}

# A: a genuinely risky REAL order - historically-high-delay seller.
# This should score normally, NOT be penalised for being "unusual".
scenarios["A: real high-risk order"] = predict_with_guardrails(
    final_model, risky_real, FEATURE_COLUMNS, NUMERIC, CATEGORICAL, stats,
    threshold=honest_threshold, risk_bucket_fn=risk_bucket,
)

# B: malformed - a required field missing, plus a structurally impossible negative value
malformed = dict(risky_real)
del malformed["total_freight"]
malformed["total_price"] = -50.0
scenarios["B: malformed order"] = predict_with_guardrails(
    final_model, malformed, FEATURE_COLUMNS, NUMERIC, CATEGORICAL, stats,
    threshold=honest_threshold, risk_bucket_fn=risk_bucket,
)

# C: well-formed but physically implausible - should be scored, flagged low-confidence
extreme = dict(risky_real)
extreme["seller_customer_distance_km"] = 50000.0
extreme["total_order_value"] = 999999.0
scenarios["C: out-of-distribution order"] = predict_with_guardrails(
    final_model, extreme, FEATURE_COLUMNS, NUMERIC, CATEGORICAL, stats,
    threshold=honest_threshold, risk_bucket_fn=risk_bucket,
)

# D: a rate feature above 1.0 - structurally impossible, should be REJECTED
impossible = dict(risky_real)
impossible["seller_historical_delay_rate"] = 1.4
scenarios["D: impossible rate value"] = predict_with_guardrails(
    final_model, impossible, FEATURE_COLUMNS, NUMERIC, CATEGORICAL, stats,
    threshold=honest_threshold, risk_bucket_fn=risk_bucket,
)

for name, result in scenarios.items():
    print(f"\n{name}")
    for k, v in result.items():
        print(f"   {k}: {v}")


A: real high-risk order
   status: SCORED
   probability: 0.7462
   above_threshold: True
   confidence: NORMAL
   warnings: []
   risk_level: HIGH

B: malformed order
   status: REJECTED
   reject_reasons: ["Missing required field(s): ['total_freight']", "'total_price'=-50 is negative, which never occurs in training data for this field."]
   probability: None
   risk_level: None
   confidence: None

C: out-of-distribution order
   status: SCORED
   probability: 0.7454
   above_threshold: True
   confidence: LOW_CONFIDENCE
   warnings: ["'total_order_value'=999999 is outside the range seen in training [9.59, 13664.08] -- the model is extrapolating, so this prediction may be less reliable.", "'seller_customer_distance_km'=50000 is outside the range seen in training [0.00, 3399.19] -- the model is extrapolating, so this prediction may be less reliable."]
   risk_level: HIGH

D: impossible rate value
   status: REJECTED
   reject_reasons: ["'seller_historical_delay_rate'=1.4 is a rate fe

- **A** scores normally at NORMAL confidence - a genuinely risky order
  is not penalised just for being unusual.
- **B** is REJECTED outright with the specific reasons, rather than
  silently imputed into a confident-looking (and wrong) prediction.
- **C** is still scored (refusing anything even slightly unusual would
  make the system useless) but flagged LOW_CONFIDENCE with the exact
  fields responsible.
- **D** is REJECTED - a rate feature above 1.0 is not a matter of
  opinion, it's a data error.

## 4. Save the corrected, honest results

In [7]:
metrics = json.load(open("../dashboard/model_metrics.json"))
metrics["selected_threshold"] = float(honest_threshold)
metrics["selected_threshold_note"] = (
    "Corrected in notebook 07: this threshold is selected on a validation "
    "slice of TRAINING data, not the test set. An earlier version of this "
    "notebook selected the threshold directly on the test set (data "
    "leakage) - see notebook 07 section 2 for the before/after comparison."
)
metrics["xgboost_tuned"] = {k: float(v) for k, v in res_honest.items() if k in ("precision", "recall", "f1", "roc_auc")}
metrics["xgboost_tuned_leaky_threshold_for_reference"] = {
    k: float(v) for k, v in res_leaky.items() if k in ("precision", "recall", "f1", "roc_auc")
}
metrics["data_validation_verdict_on_current_raw_data"] = report["verdict"]

with open("../dashboard/model_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Updated dashboard/model_metrics.json with the leak-free threshold and metrics.")
print("Saved models/reference_stats.json for inference-time guardrails.")
print(f"\nFinal reported numbers (honest): precision={res_honest['precision']:.3f} "
      f"recall={res_honest['recall']:.3f} f1={res_honest['f1']:.3f} roc_auc={res_honest['roc_auc']:.3f} "
      f"@ threshold={honest_threshold:.3f}")

Updated dashboard/model_metrics.json with the leak-free threshold and metrics.
Saved models/reference_stats.json for inference-time guardrails.

Final reported numbers (honest): precision=0.311 recall=0.483 f1=0.378 roc_auc=0.804 @ threshold=0.618
